# 类继承关系
```mermaid
classDiagram
    class Records
    class Ranges
    class Orders
    class Trades
    class EntryTrades
    class ExitTrades
    class Positions
    
    %% 继承关系
    Records <|-- Ranges
    Records <|-- Orders
    Ranges <|-- Trades
    Trades <|-- EntryTrades
    Trades <|-- ExitTrades
    Trades <|-- Positions
```

# class Trades(Ranges)
交易记录类。除了继承 `Ranges`，还拥有属性 `_close`，用于记录参考价。

## `__init__`
```python
def __init__(self,
                wrapper: ArrayWrapper,
                records_arr: tp.RecordArray,
                close: tp.ArrayLike,
                **kwargs) -> None:
    Ranges.__init__(
        self,
        wrapper,
        records_arr,
        close=close,
        **kwargs
    )
    self._close = close
```

## indexing_func
对 `Trades` 对象执行索引操作。
- 调用 `Ranges` 的 `indexing_func_meta` 获取索引元数据
- 根据列索引对参考价格数据 `_close` 进行相应的切片
- 创建新的 `Trades` 对象

```python
def indexing_func(self: TradesT, pd_indexing_func: tp.PandasIndexingFunc, **kwargs) -> TradesT:
    new_wrapper, new_records_arr, group_idxs, col_idxs = \
        Ranges.indexing_func_meta(self, pd_indexing_func, **kwargs)
    if self.close is not None:
        new_close = new_wrapper.wrap(to_2d_array(self.close)[:, col_idxs], group_by=False)
    else:
        new_close = None
    return self.replace(
        wrapper=new_wrapper,
        records_arr=new_records_arr,
        close=new_close
    )
```

## winning/losing
过滤出盈利/亏损的交易记录并创建新的 `Trades` 实例返回。
```python
@cached_property
def winning(self: TradesT) -> TradesT:
    filter_mask = self.values['pnl'] > 0.
    return self.apply_mask(filter_mask)

@cached_property
def losing(self: TradesT) -> TradesT:
    filter_mask = self.values['pnl'] < 0.
    return self.apply_mask(filter_mask)
```

## win_rate
计算 $胜率 = \frac{盈利交易数}{总交易数}$
```python
@cached_method
def win_rate(self, group_by: tp.GroupByLike = None,
                wrap_kwargs: tp.KwargsLike = None) -> tp.MaybeSeries:
    win_count = to_1d_array(self.winning.count(group_by=group_by))
    total_count = to_1d_array(self.count(group_by=group_by))
    with np.errstate(divide='ignore', invalid='ignore'):
        win_rate = win_count / total_count
    wrap_kwargs = merge_dicts(dict(name_or_index='win_rate'), wrap_kwargs)
    return self.wrapper.wrap_reduced(win_rate, group_by=group_by, **wrap_kwargs)
```

## profit_factor
计算 
$$盈利因子 = \frac{总盈利金额}{{\left| 总亏损金额 \right|}}$$
```python
@cached_method
def profit_factor(self, group_by: tp.GroupByLike = None,
                    wrap_kwargs: tp.KwargsLike = None) -> tp.MaybeSeries:
    total_win = to_1d_array(self.winning.pnl.sum(group_by=group_by))
    total_loss = to_1d_array(self.losing.pnl.sum(group_by=group_by))

    has_values = to_1d_array(self.count(group_by=group_by)) > 0
    total_win[np.isnan(total_win) & has_values] = 0.
    total_loss[np.isnan(total_loss) & has_values] = 0.

    with np.errstate(divide='ignore', invalid='ignore'):
        profit_factor = total_win / np.abs(total_loss)
    wrap_kwargs = merge_dicts(dict(name_or_index='profit_factor'), wrap_kwargs)
    return self.wrapper.wrap_reduced(profit_factor, group_by=group_by, **wrap_kwargs)
```

## expectancy
计算 
$$期望收益 = 胜率 \cdot 平均盈利 - \left( {1 - 胜率} \right) \cdot \left| 平均亏损 \right|$$
```python
@cached_method
def expectancy(self, group_by: tp.GroupByLike = None,
                wrap_kwargs: tp.KwargsLike = None) -> tp.MaybeSeries:
    win_rate = to_1d_array(self.win_rate(group_by=group_by))
    avg_win = to_1d_array(self.winning.pnl.mean(group_by=group_by))
    avg_loss = to_1d_array(self.losing.pnl.mean(group_by=group_by))

    has_values = to_1d_array(self.count(group_by=group_by)) > 0
    avg_win[np.isnan(avg_win) & has_values] = 0.
    avg_loss[np.isnan(avg_loss) & has_values] = 0.

    expectancy = win_rate * avg_win - (1 - win_rate) * np.abs(avg_loss)
    wrap_kwargs = merge_dicts(dict(name_or_index='expectancy'), wrap_kwargs)
    return self.wrapper.wrap_reduced(expectancy, group_by=group_by, **wrap_kwargs)
```

## sqn
计算
$$系统质量数 = \sqrt {交易次数}  \cdot \frac{平均盈利}{盈利标准差}$$
```python
@cached_method
def sqn(self, group_by: tp.GroupByLike = None,
        wrap_kwargs: tp.KwargsLike = None) -> tp.MaybeSeries:
    count = to_1d_array(self.count(group_by=group_by))
    pnl_mean = to_1d_array(self.pnl.mean(group_by=group_by))
    pnl_std = to_1d_array(self.pnl.std(group_by=group_by))
    sqn = np.sqrt(count) * pnl_mean / pnl_std
    wrap_kwargs = merge_dicts(dict(name_or_index='sqn'), wrap_kwargs)
    return self.wrapper.wrap_reduced(sqn, group_by=group_by, **wrap_kwargs)
```

# class EntryTrades(Trades)
入场交易类。

方法 `from_orders`：从 `orders.values` 中提取完整持仓过程中的入场记录，然后创建并返回新的 `EntryTrades` 对象。

## 源码
```python
@override_field_config(entry_trades_field_config)
class EntryTrades(Trades):
    @classmethod
    def from_orders(cls: tp.Type[EntryTradesT],
                    orders: Orders,
                    close: tp.Optional[tp.ArrayLike] = None,
                    attach_close: bool = True,
                    **kwargs) -> EntryTradesT:
        if close is None:
            close = orders.close
        # 从订单 orders.values 中提取完整持仓过程（持仓——>平仓）中的入场记录到 trade_records_arr
        trade_records_arr = nb.get_entry_trades_nb(
            orders.values,
            to_2d_array(close),
            orders.col_mapper.col_map
        )
        return cls(orders.wrapper, trade_records_arr, close=close if attach_close else None, **kwargs)
```

# class ExitTrades(Trades)
出场交易类。

方法 `from_orders`：从 `orders.values` 中提取完整持仓过程中的出场记录，然后创建并返回新的 `ExitTrades` 对象。

### 源码
```python
@override_field_config(exit_trades_field_config)
class ExitTrades(Trades):
    @classmethod
    def from_orders(cls: tp.Type[ExitTradesT],
                    orders: Orders,
                    close: tp.Optional[tp.ArrayLike] = None,
                    attach_close: bool = True,
                    **kwargs) -> ExitTradesT:
        if close is None:
            close = orders.close
        trade_records_arr = nb.get_exit_trades_nb(
            orders.values,
            to_2d_array(close),
            orders.col_mapper.col_map
        )
        return cls(orders.wrapper, trade_records_arr, close=close if attach_close else None, **kwargs)
```

# class Positions(Trades)
持仓记录类。

方法 `from_orders`：聚合 `trades.values` 中的交易记录，然后创建并返回新的 `ExitTrades` 对象。

### 源码
```python
@override_field_config(positions_field_config)
class Positions(Trades):
    @property
    def field_config(self) -> Config:
        return self._field_config

    @classmethod
    def from_trades(cls: tp.Type[PositionsT],
                    trades: Trades,
                    close: tp.Optional[tp.ArrayLike] = None,
                    attach_close: bool = True,
                    **kwargs) -> PositionsT:
        if close is None:
            close = trades.close
        position_records_arr = nb.get_positions_nb(trades.values, trades.col_mapper.col_map)
        return cls(trades.wrapper, position_records_arr, close=close if attach_close else None, **kwargs)
```